# ۱. فهرست داده Aurora-BP

این نوت‌بوک دو آرشیو پروتکل را جدا می‌خواند و چیزی را استخراج نمی‌کند.

- `/content/drive/MyDrive/aurora_dataset`
- `measurements_auscultatory (1).zip`
- `measurements_oscillometric.zip`

جدول‌های کوچک از پوشه یا از داخل ZIP خوانده می‌شوند. از هر آرشیو فقط بیست سطرِ یک فایل موج برای سنجش ستون‌ها و گام زمانی باز می‌شود.


In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('خارج از کولب؛ اتصال درایو رد شد.')


Mounted at /content/drive


In [3]:
import base64
from pathlib import Path
import sys

ROOT = Path('/content/aurorabp_reproduction')
package = ROOT / 'src'
package.mkdir(parents=True, exist_ok=True)
(package / '__init__.py').write_bytes(base64.b64decode('IiIiQXVyb3JhLUJQIHJlcHJvZHVjdGlvbiBwYWNrYWdlLiBBdXRob3Igc2V0dGluZ3MgYXJlIG5vdCBpbmZlcnJlZCBzaWxlbnRseS4iIiIKCl9fdmVyc2lvbl9fID0gIjAuMS4wIgo='))
(package / 'inventory.py').write_bytes(base64.b64decode('...'))
sys.path.insert(0, str(ROOT))
print(ROOT)


/content/aurorabp_reproduction


In [4]:
from pathlib import Path
import sys

candidates = [
    Path('/content/aurorabp_reproduction'),
    Path('/content/drive/MyDrive/aurorabp_reproduction'),
    Path.cwd(),
    Path.cwd().parent,
]
PROJECT = next(path for path in candidates if (path / 'src' / 'inventory.py').exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print(PROJECT)


/content/aurorabp_reproduction


In [5]:
DATASET_DIR = '/content/drive/MyDrive/aurora_dataset'
AUSCULTATORY_ZIP = '/content/drive/MyDrive/aurora_dataset/measurements_auscultatory (1).zip'
OSCILLOMETRIC_ZIP = '/content/drive/MyDrive/aurora_dataset/measurements_oscillometric.zip'
OUTPUT_DIR = Path('/content/aurorabp_outputs')
if not Path('/content').exists():
    OUTPUT_DIR = PROJECT / 'outputs' / 'inventory'


In [6]:
import shutil
from pathlib import Path

for label, path in (
    ('auscultatory', AUSCULTATORY_ZIP),
    ('oscillometric', OSCILLOMETRIC_ZIP),
):
    file_path = Path(path)
    if file_path.is_file():
        print(f'{label}: {file_path.stat().st_size / (1024 ** 3):.2f} GB')
    else:
        print(f'{label}: فایل پیدا نشد -> {path}')

usage = shutil.disk_usage('/content' if Path('/content').exists() else '.')
free_gb = usage.free / (1024 ** 3)
print(f'فضای آزاد: {free_gb:.1f} GB')
if free_gb < 40:
    print('استخراج کامل ۳۰ گیگابایتی روی این دیسک جا نمی‌شود. همین نوت‌بوک فقط فهرست می‌سازد.')
else:
    print('فضای دیسک برای استخراج بعدی کافی به نظر می‌رسد، ولی این نوت‌بوک استخراج را شروع نمی‌کند.')


auscultatory: 1.51 GB
oscillometric: 4.67 GB
فضای آزاد: 107.6 GB
فضای دیسک برای استخراج بعدی کافی به نظر می‌رسد، ولی این نوت‌بوک استخراج را شروع نمی‌کند.


In [7]:
from src.inventory import inspect_collection, write_inventory

report = inspect_collection(
    DATASET_DIR,
    [AUSCULTATORY_ZIP, OSCILLOMETRIC_ZIP],
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
destination = OUTPUT_DIR / 'dataset_inventory.json'
write_inventory(report, str(destination))
print(destination)
report


/content/aurorabp_outputs/dataset_inventory.json


{'dataset_dir': '/content/drive/MyDrive/aurora_dataset',
 'zip_paths': ['/content/drive/MyDrive/aurora_dataset/measurements_auscultatory (1).zip',
  '/content/drive/MyDrive/aurora_dataset/measurements_oscillometric.zip'],
 'open_errors': [],
 'table_origins': {'participants.tsv': '/content/drive/MyDrive/aurora_dataset',
  'measurements_auscultatory.tsv': '/content/drive/MyDrive/aurora_dataset',
  'measurements_oscillometric.tsv': '/content/drive/MyDrive/aurora_dataset',
  'features.tsv': '/content/drive/MyDrive/aurora_dataset'},
 'missing_tables': [],
 'archives': [{'source': '/content/drive/MyDrive/aurora_dataset/measurements_auscultatory (1).zip',
   'member_count': 11427,
   'archive_root': 'measurements_auscultatory',
   'n_waveform_files': 11427,
   'n_participant_folders': 672,
   'phases': {'initial': 8080, 'return': 3347},
   'example_members': ['measurements_auscultatory/a000/a000.initial.Calibration_start_1.tsv',
    'measurements_auscultatory/a000/a000.initial.Calibration_st